# imports

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import mlflow
import mlflow.tensorflow

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

c:\Users\Manindra\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# config

In [5]:
ASSET = 'BTC'
INTERVAL = '4h'
SEQUENCE_LENGTH = 20
N_FEATURES = 5
# model hyperparameters (same as old LSTM for fair comparison)
LSTM_UNITS_1 = 64
LSTM_UNITS_2 = 32
DENSE_UNITS = 16
DROPOUT_1 = 0.4
DROPOUT_2 = 0.2
LEARNING_RATE = 0.0001
EPOCHS = 50
BATCH_SIZE = 64
EARLY_STOP_PATIENCE = 15

DATA_DIR = '../../../data/processed/'

X_TRAIN_PATH = os.path.join(DATA_DIR, f'X_train_{ASSET.lower()}_{INTERVAL}_lstm_raw.npy')
X_TEST_PATH  = os.path.join(DATA_DIR, f'X_test_{ASSET.lower()}_{INTERVAL}_lstm_raw.npy')
Y_TRAIN_PATH = os.path.join(DATA_DIR, f'y_train_{ASSET.lower()}_{INTERVAL}_lstm_raw.parquet')
Y_TEST_PATH  = os.path.join(DATA_DIR, f'y_test_{ASSET.lower()}_{INTERVAL}_lstm_raw.parquet')

print(f" Asset: {ASSET} | Interval: {INTERVAL}")
print(f" Sequence length: {SEQUENCE_LENGTH} candles | Features: {N_FEATURES} log-returns")
print(f" Input files:")
print(f"   X_train: {X_TRAIN_PATH}")
print(f"   X_test:  {X_TEST_PATH}")
print(f"   y_train: {Y_TRAIN_PATH}")
print(f"   y_test:  {Y_TEST_PATH}")


 Asset: BTC | Interval: 4h
 Sequence length: 20 candles | Features: 5 log-returns
 Input files:
   X_train: ../../../data/processed/X_train_btc_4h_lstm_raw.npy
   X_test:  ../../../data/processed/X_test_btc_4h_lstm_raw.npy
   y_train: ../../../data/processed/y_train_btc_4h_lstm_raw.parquet
   y_test:  ../../../data/processed/y_test_btc_4h_lstm_raw.parquet


# mlflow setup

In [3]:
mlflow.set_tracking_uri("http://localhost:5000")

EXPERIMENT_NAME = f"{ASSET}_{INTERVAL}_deep_learning"

try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
    print(f"Created new MLflow experiment: '{EXPERIMENT_NAME}'")
except:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
    print(f"Using existing MLflow experiment: '{EXPERIMENT_NAME}'")

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"   Experiment ID: {experiment_id}")

Created new MLflow experiment: 'BTC_4h_deep_learning'
   Experiment ID: 21


# load pre-built sequences from fe notebook

In [6]:
X_train = np.load(X_TRAIN_PATH)
X_test  = np.load(X_TEST_PATH)

y_train_df = pd.read_parquet(Y_TRAIN_PATH)
y_test_df  = pd.read_parquet(Y_TEST_PATH)

y_train = y_train_df['target_direction'].values
y_test  = y_test_df['target_direction'].values

print(f"data loaded successfully")
print(f"   X_train shape: {X_train.shape}  (samples, timesteps, features)")
print(f"   X_test  shape: {X_test.shape}")
print(f"   y_train shape: {y_train.shape}")
print(f"   y_test  shape: {y_test.shape}")


data loaded successfully
   X_train shape: (10619, 20, 5)  (samples, timesteps, features)
   X_test  shape: (2655, 20, 5)
   y_train shape: (10619,)
   y_test  shape: (2655,)
